---
## Environment Setup & Data Verification
Ensure all required datasets (`transactions.csv`, `refunds.xlsx`, `customer_complaints.json`) are present in the directory and create the `phonepe_reports` output folder.



In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

DATA_DIR = Path('.')
required = ['transactions.csv', 'refunds.xlsx', 'customer_complaints.json']

if not all((DATA_DIR / name).exists() for name in required):
    try:
        from google.colab import files
        print('Upload these three files:', ', '.join(required))
        files.upload()
    except ImportError:
        pass

assert all((DATA_DIR / name).exists() for name in required), 'Missing one or more input datasets.'
OUTPUT_DIR = Path('phonepe_reports')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Inputs available:', required)



---
## Milestone 1: Load and Assess Transaction Data

### Workplace Requirement:
The operations manager needs a baseline view of payment transactions before analyzing disputes.

### Your Tasks:
1. Load `transactions.csv` into a DataFrame named `transactions_raw`.
2. Inspect the shape and display the first 5 records.
3. Check column data types (`dtypes`).
4. Find the number of unique transactions (`transaction_id`) and unique customers (`customer_id`).
5. Check for missing values across all columns (`isna().sum()`).

*Expected shape:* `(5032, 9)`, *Unique transactions:* `5000`, *Missing amount values:* `28`.



In [ ]:
# Milestone 1 Implementation
transactions_raw = pd.read_csv(DATA_DIR / 'transactions.csv')

print('Rows/columns:', transactions_raw.shape)
display(transactions_raw.head())
display(transactions_raw.dtypes.rename('data_type').to_frame())
print('Unique transactions:', transactions_raw['transaction_id'].nunique())
print('Unique customers:', transactions_raw['customer_id'].nunique())
display(transactions_raw.isna().sum().rename('missing_values').to_frame())



---
## Milestone 2: Inspect Refunds and Complaints

### Workplace Requirement:
The dispute desk needs an overview of refund activity and complaint volumes from two separate formats (Excel and JSON).

### Your Tasks:
1. Load `refunds.xlsx` using `pd.read_excel()` into `refunds_raw`.
2. Load `customer_complaints.json` using `pd.read_json()` into `complaints_raw`.
3. Check their shapes, preview the first 3 rows, and inspect missing values.
4. Check frequency distributions:
   - `refund_status` in refunds (notice casing/formatting inconsistencies like 'Pending', 'PENDING', 'pending').
   - `complaint_type` in complaints.

*Expected shapes:* Refunds: `(1575, 5)`, Complaints: `(1676, 6)`.



In [ ]:
# Milestone 2 Implementation
refunds_raw = pd.read_excel(DATA_DIR / 'refunds.xlsx')
complaints_raw = pd.read_json(DATA_DIR / 'customer_complaints.json')

for label, frame in [('Refunds', refunds_raw), ('Complaints', complaints_raw)]:
    print(label, 'shape:', frame.shape)
    display(frame.head(3))
    display(frame.isna().sum().rename('missing_values').to_frame())

display(refunds_raw['refund_status'].value_counts(dropna=False).rename('refunds').to_frame().head(12))
display(complaints_raw['complaint_type'].value_counts().rename('complaints').to_frame())



---
## Milestone 3: Clean Transaction Records

### Workplace Requirement:
Duplicate transaction identifiers, inconsistent statuses, and invalid values must be handled before producing trustworthy metrics.

### Your Tasks:
1. Deduplicate by `transaction_id` using `drop_duplicates('transaction_id')`.
2. Standardize `transaction_status` by stripping whitespace and converting to Title Case (`.str.strip().str.title()`).
3. Clean `payment_mode` by stripping whitespace (`.str.strip()`).
4. Convert `transaction_date` to datetime with `pd.to_datetime(..., errors='coerce')`.
5. Convert `amount` to numeric with `pd.to_numeric(..., errors='coerce')`.
6. Flag problematic records: create a boolean column `needs_data_review` if date is null, amount is null, or amount <= 0.

*Expected verification:*
- Removed duplicate IDs: `32` (leaving `5000` records).
- Status counts: Success: `3279`, Failed: `851`, Pending: `602`, Reversed: `268`.
- Records requiring review: `62`.



In [ ]:
# Milestone 3 Implementation
transactions_clean = transactions_raw.drop_duplicates('transaction_id').copy()

transactions_clean['transaction_status'] = transactions_clean['transaction_status'].astype('string').str.strip().str.title()
transactions_clean['payment_mode'] = transactions_clean['payment_mode'].astype('string').str.strip()
transactions_clean['transaction_date'] = pd.to_datetime(transactions_clean['transaction_date'], errors='coerce')
transactions_clean['amount'] = pd.to_numeric(transactions_clean['amount'], errors='coerce')

# Flag transactions needing review
transactions_clean['needs_data_review'] = (
    transactions_clean['transaction_date'].isna() |
    transactions_clean['amount'].isna() |
    transactions_clean['amount'].le(0)
)

print('Removed duplicated IDs:', len(transactions_raw) - len(transactions_clean))
display(transactions_clean['transaction_status'].value_counts().rename('transactions').to_frame())
print('Records requiring data review:', transactions_clean['needs_data_review'].sum())



---
## Milestone 4: Clean Issue and Refund Records

### Workplace Requirement:
Refund requests and customer complaint records must use consistent status labels and valid identifiers before being linked to transactions.

### Your Tasks:
1. Deduplicate `refunds_raw` on `refund_id` -> `refunds_clean`.
2. Standardize `refund_status` (strip, title case) and parse `refund_date` and `refund_amount`.
3. Deduplicate `complaints_raw` on `complaint_id` -> `complaints_clean`.
4. Standardize `complaint_status` (strip, title case) and parse `complaint_date`.
5. Check referential integrity: Check if any `transaction_id` in refunds or complaints does NOT exist in `transactions_clean['transaction_id']`. Flag them with boolean column `unmatched_transaction`.

*Expected verification:*
- Unique refunds: `1550` | Unmatched: `35`
- Unique complaints: `1650` | Unmatched: `30`



In [ ]:
# Milestone 4 Implementation
refunds_clean = refunds_raw.drop_duplicates('refund_id').copy()
refunds_clean['refund_status'] = refunds_clean['refund_status'].astype('string').str.strip().str.title()
refunds_clean['refund_date'] = pd.to_datetime(refunds_clean['refund_date'], errors='coerce')
refunds_clean['refund_amount'] = pd.to_numeric(refunds_clean['refund_amount'], errors='coerce')

complaints_clean = complaints_raw.drop_duplicates('complaint_id').copy()
complaints_clean['complaint_status'] = complaints_clean['complaint_status'].astype('string').str.strip().str.title()
complaints_clean['complaint_date'] = pd.to_datetime(complaints_clean['complaint_date'], errors='coerce')

valid_tx = set(transactions_clean['transaction_id'])
refunds_clean['unmatched_transaction'] = ~refunds_clean['transaction_id'].isin(valid_tx)
complaints_clean['unmatched_transaction'] = ~complaints_clean['transaction_id'].isin(valid_tx)

print('Unique refunds:', len(refunds_clean), '| Unmatched:', refunds_clean['unmatched_transaction'].sum())
print('Unique complaints:', len(complaints_clean), '| Unmatched:', complaints_clean['unmatched_transaction'].sum())



---
## Milestone 5: Analyze Transaction Status and Payment Behavior

### Workplace Requirement:
The payments operations team needs failure and pending-payment patterns across payment modes and cities.

### Your Tasks:
1. `status_summary`: Group `transactions_clean` by `transaction_status`, aggregating transaction count and sum of `amount`.
2. `payment_mode_summary`: Group by `payment_mode` and `transaction_status` to see failure/pending volumes per mode.
3. `city_issue_summary`: Identify cities with the highest count of failed or pending transactions.

*Key Insights:*
- UPI accounts for the highest transaction count and highest issue count (600 Failed, 412 Pending).
- Top cities with payment issues include Pune (167), Chennai (155), Hyderabad (153).



In [ ]:
# Milestone 5 Implementation
status_summary = transactions_clean.groupby('transaction_status', dropna=False).agg(
    transaction_count=('transaction_id', 'size'),
    total_amount=('amount', 'sum')
).reset_index()

payment_mode_summary = transactions_clean.groupby(['payment_mode', 'transaction_status'], dropna=False).size().rename('transaction_count').reset_index()

city_issue_summary = (
    transactions_clean.assign(is_issue=transactions_clean['transaction_status'].isin(['Failed', 'Pending']))
    .groupby('customer_city')
    .agg(
        total_transactions=('transaction_id', 'size'),
        failed_or_pending=('is_issue', 'sum')
    )
    .sort_values('failed_or_pending', ascending=False)
    .reset_index()
)

display(status_summary)
display(payment_mode_summary.head(12))
display(city_issue_summary.head(10))



---
## Milestone 6: Connect Refund Requests to Transactions

### Workplace Requirement:
Finance needs a single transaction-level summary of refund activity, including pending and failed refund requests.

### Your Tasks:
1. Flag unresolved refunds in `refunds_clean`: `refund_unresolved = refund_status.isin(['Pending', 'Failed'])`.
2. Aggregate refunds at `transaction_id` level:
   - `refund_request_count`: total count of refund records per transaction
   - `total_refund_requested`: sum of `refund_amount`
   - `unresolved_refund_count`: sum of `refund_unresolved`
3. Left join `transactions_clean` with the aggregated refund summary on `transaction_id` (validate `one_to_one`).
4. Impute missing refund metrics with `0` (since transactions without refunds have 0 requests/amount).

*Expected verification:* Exactly `5000` transactions retained.



In [ ]:
# Milestone 6 Implementation
refunds_clean['refund_unresolved'] = refunds_clean['refund_status'].isin(['Pending', 'Failed'])

refund_agg = refunds_clean.groupby('transaction_id').agg(
    refund_request_count=('refund_id', 'size'),
    total_refund_requested=('refund_amount', 'sum'),
    unresolved_refund_count=('refund_unresolved', 'sum')
).reset_index()

transaction_refunds = transactions_clean.merge(
    refund_agg,
    on='transaction_id',
    how='left',
    validate='one_to_one'
)

for col in ['refund_request_count', 'total_refund_requested', 'unresolved_refund_count']:
    transaction_refunds[col] = transaction_refunds[col].fillna(0)

print('Transactions retained:', len(transaction_refunds))
display(refunds_clean['refund_status'].value_counts().rename('refund_count').to_frame())
display(transaction_refunds.sort_values('total_refund_requested', ascending=False).head(10))



---
## Milestone 7: Analyze Complaints and Repeat Issues

### Workplace Requirement:
The customer experience team needs visibility into complaint categories, unresolved issues, and transactions with repeat complaints.

### Your Tasks:
1. Flag unresolved complaints: `complaint_unresolved = complaint_status.isin(['Open', 'In Progress'])`.
2. Generate `complaint_type_summary` using `.value_counts()`.
3. Aggregate complaints at `transaction_id` level:
   - `complaint_count`: total complaints logged for that transaction
   - `unresolved_complaint_count`: total open or in-progress complaints
4. Identify repeat complaints: filter for transactions where `complaint_count > 1`.
5. Identify repeat dissatisfied customers: group complaints by `customer_id`.

*Expected verification:*
- Complaints breakdown: Resolved: 628, Open: 569, In Progress: 453.
- Transactions with repeat complaints (up to 4 complaints per single transaction).



In [ ]:
# Milestone 7 Implementation
complaints_clean['complaint_unresolved'] = complaints_clean['complaint_status'].isin(['Open', 'In Progress'])

complaint_type_summary = complaints_clean['complaint_type'].value_counts().rename('complaint_count').to_frame()

complaint_agg = complaints_clean.groupby('transaction_id').agg(
    complaint_count=('complaint_id', 'size'),
    unresolved_complaint_count=('complaint_unresolved', 'sum')
).reset_index()

repeat_complaints = complaint_agg.loc[complaint_agg['complaint_count'] > 1].sort_values('complaint_count', ascending=False)
customer_complaint_summary = complaints_clean.groupby('customer_id').size().sort_values(ascending=False).rename('complaint_count').to_frame()

display(complaint_type_summary)
display(complaints_clean['complaint_status'].value_counts().to_frame())
display(repeat_complaints.head(10))
display(customer_complaint_summary.head(10))



---
## Milestone 8: Build the Unified Dispute Master Table

### Workplace Requirement:
A single consolidated transaction-level view is required without multiplying rows when multiple refunds and complaints exist for the same payment.

### Your Tasks:
1. Merge `transaction_refunds` with `complaint_agg` using a left join on `transaction_id` (`validate='one_to_one'`).
2. Impute `complaint_count` and `unresolved_complaint_count` with `0`, then cast to integer.
3. Ensure refund counts are also integer typed.
4. Run validation assertions:
   - Assert `dispute_master['transaction_id'].is_unique`
   - Assert `len(dispute_master) == 5000`
   - Check shape (5000 rows, 15 columns).



In [ ]:
# Milestone 8 Implementation
dispute_master = transaction_refunds.merge(
    complaint_agg,
    on='transaction_id',
    how='left',
    validate='one_to_one'
)

for col in ['complaint_count', 'unresolved_complaint_count']:
    dispute_master[col] = dispute_master[col].fillna(0).astype(int)

for col in ['refund_request_count', 'unresolved_refund_count']:
    dispute_master[col] = dispute_master[col].astype(int)

assert dispute_master['transaction_id'].is_unique, 'Transaction ID must be unique!'
assert len(dispute_master) == transactions_clean['transaction_id'].nunique(), 'Row count must match unique transactions!'

print('Master shape:', dispute_master.shape)
display(dispute_master.head())



---
## Milestone 9: Classify Dispute Support Priority

### Workplace Requirement:
Support managers need a transparent priority rule based on failed/pending payments, unresolved refunds, and repeated/unresolved complaints.

### Priority Rules Logic:
We evaluate 3 key issue signals:
1. **`payment_issue`**: Transaction status is `'Failed'` or `'Pending'`.
2. **`refund_issue`**: `unresolved_refund_count > 0`.
3. **`complaint_issue`**: `complaint_count > 1` (repeat complaints) OR `unresolved_complaint_count > 0`.

**Total Signals:** `issue_signal_count = payment_issue + refund_issue + complaint_issue` (range: 0 to 3).

### Priority Categories:
- **High**: `issue_signal_count >= 2` (multiple failure points, severe customer impact)
- **Medium**: `issue_signal_count == 1` (isolated single issue)
- **Low**: `issue_signal_count == 0` (clean, completed transactions)

*Expected Counts:*
- High: `535`
- Medium: `1998`
- Low: `2467`



In [ ]:
# Milestone 9 Implementation
dispute_master['payment_issue'] = dispute_master['transaction_status'].isin(['Failed', 'Pending'])
dispute_master['refund_issue'] = dispute_master['unresolved_refund_count'].gt(0)
dispute_master['complaint_issue'] = dispute_master['complaint_count'].gt(1) | dispute_master['unresolved_complaint_count'].gt(0)

dispute_master['issue_signal_count'] = dispute_master[['payment_issue', 'refund_issue', 'complaint_issue']].sum(axis=1)

dispute_master['support_priority'] = np.select(
    [dispute_master['issue_signal_count'].ge(2), dispute_master['issue_signal_count'].eq(1)],
    ['High', 'Medium'],
    default='Low'
)

priority_summary = (
    dispute_master['support_priority']
    .value_counts()
    .reindex(['High', 'Medium', 'Low'], fill_value=0)
    .rename_axis('support_priority')
    .reset_index(name='transaction_count')
)

display(priority_summary)
display(dispute_master.loc[
    dispute_master['support_priority'] == 'High',
    ['transaction_id', 'transaction_status', 'unresolved_refund_count', 'complaint_count', 'support_priority']
].head(10))



---
## Milestone 10: Export Dispute Resolution Reports and Present Findings

### Workplace Requirement:
The operations manager needs a prioritized transaction report and concise executive summary for weekly review.

### Your Tasks:
1. Sort `dispute_master` by `support_priority` (High -> Medium -> Low) using a categorical order, then by `transaction_id`.
2. Export the full dispute report to `phonepe_reports/phonepe_transaction_dispute_report.csv`.
3. Export the priority summary to `phonepe_reports/phonepe_dispute_priority_summary.csv`.
4. Output key business findings.

*Expected Summary Findings:*
- Failed or pending transactions: `1453`
- Transactions with unresolved refunds: `699`
- Transactions with complaint signals: `949`
- High-priority transactions: `535`



In [ ]:
# Milestone 10 Implementation
priority_order = pd.Categorical(
    dispute_master['support_priority'],
    categories=['High', 'Medium', 'Low'],
    ordered=True
)

final_report = (
    dispute_master.assign(_priority_sort=priority_order)
    .sort_values(['_priority_sort', 'transaction_id'])
    .drop(columns='_priority_sort')
    .copy()
)

report_file = OUTPUT_DIR / 'phonepe_transaction_dispute_report.csv'
summary_file = OUTPUT_DIR / 'phonepe_dispute_priority_summary.csv'

final_report.to_csv(report_file, index=False)
priority_summary.to_csv(summary_file, index=False)

print('Saved reports to:', report_file, 'and', summary_file)
print('\n--- Executive Business Findings ---')
print('- Failed or pending transactions:', int(dispute_master['payment_issue'].sum()))
print('- Transactions with unresolved refunds:', int(dispute_master['refund_issue'].sum()))
print('- Transactions with complaint signals:', int(dispute_master['complaint_issue'].sum()))
print('- High-priority transactions:', int((dispute_master['support_priority'] == 'High').sum()))

display(final_report.head(10))

